In [59]:
import sys
import math
from collections import defaultdict
from scipy.spatial import distance
import numpy as np
import itertools

class Table:
    def __init__(self, name, age):
        self.key = name
        self.values = age

    def __str__(self):
        return self.name + str(self.age)

    def __hash__(self):
        print(hash(str(self)))
        return hash(str(self))

    def __eq__(self,other):
        return self.name == other.name and self.age==other.age

def sorted_dict_to_tuple(d):  # sorted_dicts_to_tuple takes the dictionary as input and sorts it into tuple
    return tuple(sorted(d.items()))

def all_duplicates(dicts):  # The all_duplicates function will check all the elements in the dictionary and keep track of any repeating element
    seen = set() 
    return [d for d in dicts if not (sorted_dict_to_tuple(d) in seen or seen.add(sorted_dict_to_tuple(d)))]

def get_patterns_list(height,width):
    cases = []
    for nr in range(height):
        for nc in range(width):
            cases.append((nr,nc))
            
    dist = distance.cdist(cases, cases, 'euclidean')
            
    groups = defaultdict(list)
    for key, value in np.argwhere(dist==1):
        groups[key].extend([key, value])

    groups = list(map(set,list(groups.values())))

    patterns = {}
    for k,g in enumerate(groups):
        combinations = []
        for i in range(3,6):
            c = list(itertools.combinations(g,i))
            combinations.append(c)
        combinations = list(itertools.chain(*combinations))
        combinations = [list(map(int,i)) for i in combinations if k in i]
        combinations = [[i for i in x if i != k] for x in combinations]
        patterns[k] = combinations

    return patterns

def debug(txt):
    print(txt,file=sys.stderr,flush=True)

def process_values(values,grid,zero_index,pattern_index):
    s = sum(values)
    g = grid.copy()
    state = ""
    if (s <= 6) and (0 not in values):
        for t in pattern_index:
            g.update({t:0})
        g.update({zero_index:s})
        state = "capture"
    else:
        g.update({zero_index:1})
    return (state,g)

def compute(grid,patterns):
    '''
    La fonction "compute" traite chaque grille unitairement pour évaluer les prochains coups 
    Tout d'abord, elle identifie les zéros présent dans la grille. Pour chaque index où la valeur
    est égale à zéro, on récupère l'ensemble des patterns de capture associés (les patterns sont des tableaux d'entiers),
    
    '''
    zeros_indexes = [k for k,v in grid.items() if v==0]
    if len(zeros_indexes) != 0:
        result = []
        for zi in zeros_indexes:
            pattern_indexes = patterns.get(zi)                
            #Itérer sur les patterns de captures pour un zero_index donné et retourner les grilles calculées
            pv = [process_values(list(map(lambda x : grid.get(x),pi)),grid,zi,pi) for pi in pattern_indexes]
            if "capture" in list(map(lambda x : x[0],pv)):
                pv = [x for x in pv if x[0] == "capture"]
            result += all_duplicates(list(map(lambda x : x[1],pv)))  
        return result
    else:
        return [grid]
    
def resolve(val,patterns):
    '''
    La fonction "resolve" est une fonction récursive pour appliquer le traitement sur les données.
    
    :param val: Le paramètre val est une liste de dictionnaire représentant les différentes grilles après chaque coup 
    :param patterns: La liste patterns contient l'ensemble des patterns de captures sur une grille de dimension donnée
    :return: La fonction retourne une liste de grille (dict)  
    '''
    #On récupère au sein de la fonction la variable "depth" pour monitorer le nombre de tour qu'il reste
    global depth
    #Execution de la fonction "compute" dans un map pour évaluer toutes les options de jeux sur chaque grille dans la liste "val" 
    result = list(itertools.chain(*list(map(lambda x : compute(x,patterns),val))))
    print(len(result))
    print(len(set(map(concat_dict_values,result))))
    if all(map(lambda x : 0 not in x.values(),result)):
        return result
    else:
        depth -= 1
        if depth == 0:
            return result
        else:
            return resolve(result,patterns)

def concat_dict_values(x):
    return int("".join(map(str,list(x.values()))))

update = lambda j,k: (j+k)%2**30
   
patterns = {0: [[1, 3]],
 1: [[0, 2], [0, 4], [2, 4], [0, 2, 4]],
 2: [[1, 5]],
 3: [[0, 4], [0, 6], [4, 6], [0, 4, 6]],
 4: [[1, 3],
  [1, 5],
  [1, 7],
  [3, 5],
  [3, 7],
  [5, 7],
  [1, 3, 5],
  [1, 3, 7],
  [1, 5, 7],
  [3, 5, 7],
  [1, 3, 5, 7]],
 5: [[8, 2], [8, 4], [2, 4], [8, 2, 4]],
 6: [[3, 7]],
 7: [[8, 4], [8, 6], [4, 6], [8, 4, 6]],
 8: [[5, 7]]}

init = {0: 3, 1: 0, 2: 0, 3: 3, 4: 6, 5: 2, 6: 1, 7: 0, 8: 2}
depth = 24
r = list(map(lambda x : concat_dict_values(x),resolve([init],patterns)))
r = list(itertools.accumulate(r,update))[-1]

3
3
8
6
28
16
110
40
394
75
1390
132
4848
204
16068
281
48330
380


KeyboardInterrupt: 

In [53]:
# init = {0: 0, 1: 6, 2: 0, 3: 2, 4: 2, 5: 2, 6: 1, 7: 6, 8: 1}
# depth = 20
# r = list(map(lambda x : concat_dict_values(x),resolve([init],patterns)))
# r = list(itertools.accumulate(r,update))[-1]

# if r == 322444322:
#     print("Test 1 : OK")
    
# init = {0: 5, 1: 0, 2: 6, 3: 4, 4: 5, 5: 0, 6: 0, 7: 6, 8: 4}
# depth = 20
# r = list(map(lambda x : concat_dict_values(x),resolve([init],patterns)))
# r = list(itertools.accumulate(r,update))[-1]
# if r == 951223336:
#     print("Test 2 : OK")
    
    
# init = {0: 5, 1: 5, 2: 5, 3: 0, 4: 0, 5: 5, 6: 5, 7: 5, 8: 5} 
# depth = 1
# r = list(map(lambda x : concat_dict_values(x),resolve([init],patterns)))
# r = list(itertools.accumulate(r,update))[-1]

# if r == 36379286:
#     print("Test 3 : OK")
    
    
# init = {0: 6, 1: 1, 2: 6, 3: 1, 4: 0, 5: 1, 6: 6, 7: 1, 8: 6}
# depth = 1
# r = list(map(lambda x : concat_dict_values(x),resolve([init],patterns)))
# r = list(itertools.accumulate(r,update))[-1]

# if r == 264239762:
#     print("Test 4 : OK")
    
    
# init = {0: 6, 1: 0, 2: 6, 3: 0, 4: 0, 5: 0, 6: 6, 7: 1, 8: 5}    
# depth = 8
# r = list(map(lambda x : concat_dict_values(x),resolve([init],patterns)))
# r = list(itertools.accumulate(r,update))[-1]
# if r == 76092874:
#     print("Test 5 : OK")
    
    
init = {0: 3, 1: 0, 2: 0, 3: 3, 4: 6, 5: 2, 6: 1, 7: 0, 8: 2}
depth = 24
r = list(map(lambda x : concat_dict_values(x),resolve([init],patterns)))
r = list(itertools.accumulate(r,update))[-1]
if r == 661168294:
    print("Test 6 : OK")
    

3
3
8
8
28
28
110
110
394
394
1390
1390
4848
4848
16068
16068
48330
48330
126015
126015
306119
306119
773025
773025


KeyboardInterrupt: 

In [ ]:
300
362
102

In [ ]:
compute({0: 6, 1: 1, 2: 6, 3: 0, 4: 0, 5: 0, 6: 6, 7: 1, 8: 5},patterns)

In [ ]:
{0: 6, 1: 1, 2: 6, 3: 0, 4: 1, 5: 0, 6: 6, 7: 1, 8: 5},

616
010
615

In [ ]:
616
000
615


606 616 616
020 100 001
615 615 615

In [ ]:
x = [{0: 6, 1: 1, 2: 6, 3: 1, 4: 0, 5: 0, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 1, 2: 6, 3: 0, 4: 1, 5: 0, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 0, 4: 2, 5: 0, 6: 6, 7: 0, 8: 5}, {0: 6, 1: 1, 2: 6, 3: 0, 4: 0, 5: 1, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 1, 2: 6, 3: 1, 4: 0, 5: 0, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 1, 4: 1, 5: 0, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 0, 4: 2, 5: 0, 6: 6, 7: 0, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 1, 4: 0, 5: 1, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 1, 2: 6, 3: 0, 4: 1, 5: 0, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 1, 4: 1, 5: 0, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 0, 4: 1, 5: 1, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 0, 4: 0, 5: 6, 6: 6, 7: 1, 8: 0}, {0: 6, 1: 1, 2: 6, 3: 0, 4: 0, 5: 1, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 1, 4: 0, 5: 1, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 0, 4: 1, 5: 1, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 0, 4: 2, 5: 0, 6: 6, 7: 0, 8: 5}]
set(list(map(concat_dict_values,x)))

In [ ]:
set(list(map(concat_dict_values,z)))